In [1]:
from pathlib import Path
import os
# The project folder is the folder that contains data/, notebooks/, results/ ...
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROJECT_DIR = Path(os.environ.get('PROJECT_DIR', PROJECT_DIR))
print('Project folder:', PROJECT_DIR)
assert (PROJECT_DIR / 'data').exists(), 'Project folder not found - open this notebook from the notebooks folder of the project (see your Steps file)'


Project folder: /Users/pranav/Documents/game-chat-toxicity-ml 


In [2]:
import re, html, json, time
from collections import Counter
import numpy as np
import pandas as pd
 
SEED = 42
RAW_DIR   = PROJECT_DIR / 'data' / 'raw'
OUT_DIR   = PROJECT_DIR / 'data' / 'processed'
BATCH_DIR = PROJECT_DIR / 'data' / 'batches'
for d in (OUT_DIR, BATCH_DIR): d.mkdir(parents=True, exist_ok=True)
 
LOL_FILE, DOTA_FILE = 'chatlogs.xlsx', 'dota2_chat_messages.xlsx'
N_TO_LABEL  = 3400      # messages sent for labelling (some will be skipped as "X", so we keep a safety margin)
BATCH_SIZE  = 100       # messages per paste into the chat AI
CORPUS_SIZE = 200_000   # unlabelled English messages for word2vec training
print(sorted(p.name for p in RAW_DIR.iterdir()))


['chatlogs.xlsx', 'dota2_chat_messages.xlsx']


In [3]:
def pick_col(columns, options):
    for o in options:
        if o in columns: return o
    raise KeyError(f'None of {options} found. Columns are: {list(columns)}')
 
t0 = time.time()
head = pd.read_excel(RAW_DIR / LOL_FILE, nrows=3)
lol_text, lol_reason = pick_col(head.columns, ['message','text','Content']), pick_col(head.columns, ['most_common_report_reason'])
lol = pd.read_excel(RAW_DIR / LOL_FILE, usecols=[lol_text, lol_reason]).rename(columns={lol_text:'text', lol_reason:'reason'})
print('LoL rows :', len(lol), f'({time.time()-t0:.0f}s)')
 
head = pd.read_excel(RAW_DIR / DOTA_FILE, nrows=3)
dota_text = pick_col(head.columns, ['text','message','Content'])
dota = pd.read_excel(RAW_DIR / DOTA_FILE, usecols=[dota_text]).rename(columns={dota_text:'text'})
dota['reason'] = None
print('Dota rows:', len(dota), f'({time.time()-t0:.0f}s)')


LoL rows : 1048575 (37s)
Dota rows: 1048575 (58s)


In [4]:
def fix_mojibake(s):
    try:
        return s.encode('mac_roman').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        return s
 
CTRL = re.compile(r'[\x00-\x1f\x7f]')
def basic_clean(s):
    s = html.unescape(str(s))
    s = CTRL.sub(' ', s)
    return re.sub(r'\s+', ' ', s).strip()
 
for d, name in ((lol, 'LoL'), (dota, 'Dota2')):
    d['text'] = d['text'].fillna('').astype(str).map(fix_mojibake).map(basic_clean)
    d['source'] = name
 
df = pd.concat([lol, dota], ignore_index=True)[['text', 'source', 'reason']]
del lol, dota
print('Total messages:', len(df))
print('Garble fix check ->', fix_mojibake('–ª–∞–¥–Ω–æ –≥–≥'))


Total messages: 2097150
Garble fix check -> ладно гг


In [5]:
df = df[(df.text.str.len() >= 2) & (df.text.str.len() <= 200)]
df = df[~df.text.str.contains(r'https?://|www\.', case=False, regex=True)]
latin = df.text.str.count(r'[A-Za-z]')
alpha = df.text.str.count(r'[^\W\d_]')
df = df[(latin >= 2) & (latin / alpha.clip(lower=1) >= 0.85)].copy()
print('after script filter:', len(df))
 
tok = lambda s: re.findall(r"[a-z']+", s.lower())
cnt = Counter()
for s in df[df.source == 'LoL'].text.sample(min(400_000, (df.source == 'LoL').sum()), random_state=SEED):
    cnt.update(tok(s))
english_vocab = {w for w, c in cnt.items() if c >= 25}
print('English-like vocabulary size:', len(english_vocab))
 
def english_ratio(s):
    t = tok(s)
    return sum(w in english_vocab for w in t) / len(t) if t else 0.0
 
df['low'] = df.text.str.lower()
df = df.drop_duplicates('low')
print('after removing duplicates:', len(df))
df['eng'] = df.text.map(english_ratio)
df = df[df.eng >= 0.6].drop(columns='eng').reset_index(drop=True)
print('English-like unique messages:', len(df), df.source.value_counts().to_dict())


after script filter: 1604386
English-like vocabulary size: 2474
after removing duplicates: 806905
English-like unique messages: 605771 {'LoL': 466858, 'Dota2': 138913}


In [7]:
LEXICON = re.compile(r"\b(f+u+c+k\w*|shit\w*|bitch\w*|cunt\w*|dick\w*|asshole|idiot\w*|retard\w*|stupid|dumb\w*|moron\w*|"
                     r"noob\w*|n00b\w*|trash|garbage|useless|worthless|loser\w*|feeder\w*|feeding|report\w*|uninstall|kys|"
                     r"cancer\w*|autis\w*|fag\w*|gay|nigg\w*|slut\w*|whore\w*|bastard|scum|braindead|pathetic|shut up|stfu|gtfo|"
                     r"suck\w*|inting|ez|ezz+|rekt|kid|kids|bot|bots)\b")
STRONG = re.compile(r"\b(f+u+c+k+\w*|fck|fuk|retard\w*|cunt\w*|fag\w*|nigg\w*|kys|whore\w*|slut\w*|bitch\w*|cancer\w*|autis\w*|asshole|dickhead|"
                    r"motherfuck\w*|shut the fuck|piece of shit|dumbass|moron\w*|idiot\w*|mongol\w*|spastic|rape\w*)\b")
df['strong'] = df.low.str.contains(STRONG)
df['lex'] = df.low.str.contains(LEXICON) & ~df.strong
print('strong-word messages:', int(df.strong.sum()), '| milder flame-word messages:', int(df.lex.sum()))
 
TOXIC_REASONS = {'Verbal Abuse', 'Offensive Language', 'Negative Attitude'}
 
def take(frame, n, label):
    """take n rows, half from each source when possible"""
    parts = []
    per = n // 2
    for src in ('LoL', 'Dota2'):
        sub = frame[frame.source == src]
        parts.append(sub.sample(min(per, len(sub)), random_state=SEED))
    got = pd.concat(parts)
    if len(got) < n:                       # top up if one source ran short
        rest = frame.drop(got.index)
        got = pd.concat([got, rest.sample(min(n - len(got), len(rest)), random_state=SEED)])
    got = got.copy(); got['group'] = label
    return got
 
g0 = take(df[df.strong], 600, 'strong')
g1 = take(df[df.lex], 800, 'lexicon')
used = g0.index.union(g1.index)
pool2 = df[(~df.strong) & (~df.lex) & (df.source == 'LoL') & (df.reason.isin(TOXIC_REASONS)) & ~df.index.isin(used)]
g2 = pool2.sample(400, random_state=SEED).copy(); g2['group'] = 'toxic_case'
used = used.union(g2.index)
g3 = take(df[~df.index.isin(used) & ~df.strong & ~df.lex], N_TO_LABEL - len(g0) - len(g1) - len(g2), 'random')
 
cand = pd.concat([g0, g1, g2, g3]).sample(frac=1, random_state=SEED).reset_index()
cand = cand.rename(columns={'index': 'pool_index'})
cand.insert(0, 'id', np.arange(1, len(cand) + 1))
cand = cand[['id', 'text', 'source', 'group', 'pool_index']]
print(cand.group.value_counts().to_dict(), cand.source.value_counts().to_dict())
cand.sample(12, random_state=1)


/var/folders/vh/t_7crt953gg9zrwg9bk_78n80000gn/T/ipykernel_56688/390351880.py:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['strong'] = df.low.str.contains(STRONG)
/var/folders/vh/t_7crt953gg9zrwg9bk_78n80000gn/T/ipykernel_56688/390351880.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['lex'] = df.low.str.contains(LEXICON) & ~df.strong


strong-word messages: 26174 | milder flame-word messages: 68203
{'random': 1600, 'lexicon': 800, 'strong': 600, 'toxic_case': 400} {'LoL': 1900, 'Dota2': 1500}


,id,text,source,group,pool_index
605,606,guy stop troll can win easy,LoL,random,411166
3359,3360,wait i think chatbots are better at this,Dota2,random,532892
2440,2441,idiot ... the turret,LoL,strong,93424
1661,1662,also that mirana pls,Dota2,random,470098
3312,3313,and now we lose bot!,LoL,lexicon,39839
2361,2362,not by fizz,LoL,random,14670
2912,2913,trash fuck,Dota2,strong,509253
2689,2690,pls report storm,Dota2,lexicon,487798
2809,2810,ahahha SION ahaha,LoL,random,25511
1905,1906,olaf this 2 are friends,LoL,toxic_case,284532


In [8]:
cand[['id', 'text', 'source', 'group']].to_csv(OUT_DIR / 'candidates_to_label.csv', index=False, encoding='utf-8')
 
# unlabelled corpus for word2vec: English messages that are NOT in the labelling set (no leakage)
rest = df.drop(index=cand.pool_index)
corpus = rest.sample(min(CORPUS_SIZE, len(rest)), random_state=SEED)[['text']]
corpus.to_csv(OUT_DIR / 'unlabelled_corpus.csv', index=False, encoding='utf-8')
 
stats = {
    'english_unique_messages': int(len(df)),
    'by_source': {k: int(v) for k, v in df.source.value_counts().items()},
    'candidates': int(len(cand)),
    'candidates_by_group': {k: int(v) for k, v in cand.group.value_counts().items()},
    'candidates_by_source': {k: int(v) for k, v in cand.source.value_counts().items()},
    'corpus_size': int(len(corpus)),
}
json.dump(stats, open(OUT_DIR / 'prep_stats.json', 'w'), indent=2)
print(json.dumps(stats, indent=2))



{
  "english_unique_messages": 605771,
  "by_source": {
    "LoL": 466858,
    "Dota2": 138913
  },
  "candidates": 3400,
  "candidates_by_group": {
    "random": 1600,
    "lexicon": 800,
    "strong": 600,
    "toxic_case": 400
  },
  "candidates_by_source": {
    "LoL": 1900,
    "Dota2": 1500
  },
  "corpus_size": 200000
}


In [9]:
PROMPT = 'You are helping label in-game chat messages (Dota 2 and League of Legends) for a university machine-learning project.\nLabel EACH message on its own (you cannot see the rest of the conversation) with exactly ONE code:\n\n0 = NEUTRAL / POSITIVE: normal game talk, tactics, questions, jokes, thanks, praise, "gg", "gl hf", "ty", apologies, friendly banter, and exclamations that are not aimed at anyone ("holy shit", "wow").\n1 = NEGATIVE ATTITUDE: complaining, blaming, sarcasm, mocking someone\'s skill or play, passive-aggressive remarks, giving up ("this game is lost"), mild insults such as "noob", "trash", "useless", "feeder", demanding reports ("report him"), taunts like "ez", condescending remarks. NO slurs and no strong personal abuse.\n2 = DEROGATORY: strong insults aimed at a person or group: slurs (racist, sexist, homophobic, ableist), "retard", "cunt", "faggot", sexual or family insults, wishing harm or death ("kys", "i hope you die"), threats, heavy profanity aimed at someone ("fuck you", "shut the fuck up").\nX = SKIP: not English, unreadable, random keys or a single symbol, or you truly cannot tell.\n\nExamples:\nreport for unskilled player is useless -> 1\nggwp -> 0\nsuch a feeder..xD -> 1\nshut up faggot -> 2\ntotally noob team -> 1\ngo play barbie like a girl -> 2\nholy shit -> 0\nstupid as last pickin pudge -> 1\nrussian cuck -> 2\nthx -> 0\nthis game is shit -> 1\nnah -> 0\nxxkqpz -> X\n\nReply with ONLY one line per message in the form  id,label  (for example  14,0). No explanations, no extra text, no headings.'
(OUT_DIR / 'labelling_prompt.txt').write_text(PROMPT, encoding='utf-8')
 
for f in BATCH_DIR.glob('batch_*.txt'): f.unlink()
n_batches = int(np.ceil(len(cand) / BATCH_SIZE))
for b in range(n_batches):
    chunk = cand.iloc[b * BATCH_SIZE:(b + 1) * BATCH_SIZE]
    lines = [f'{r.id}: {r.text}' for r in chunk.itertuples()]
    body = PROMPT + '\n\nMESSAGES:\n' + '\n'.join(lines)
    (BATCH_DIR / f'batch_{b+1:03d}.txt').write_text(body, encoding='utf-8')
print(f'Created {n_batches} batch files in {BATCH_DIR}')
print((BATCH_DIR / 'batch_001.txt').read_text(encoding='utf-8')[:1800])


Created 34 batch files in /Users/pranav/Documents/game-chat-toxicity-ml /data/batches
You are helping label in-game chat messages (Dota 2 and League of Legends) for a university machine-learning project.
Label EACH message on its own (you cannot see the rest of the conversation) with exactly ONE code:

0 = NEUTRAL / POSITIVE: normal game talk, tactics, questions, jokes, thanks, praise, "gg", "gl hf", "ty", apologies, friendly banter, and exclamations that are not aimed at anyone ("holy shit", "wow").
1 = NEGATIVE ATTITUDE: complaining, blaming, sarcasm, mocking someone's skill or play, passive-aggressive remarks, giving up ("this game is lost"), mild insults such as "noob", "trash", "useless", "feeder", demanding reports ("report him"), taunts like "ez", condescending remarks. NO slurs and no strong personal abuse.
2 = DEROGATORY: strong insults aimed at a person or group: slurs (racist, sexist, homophobic, ableist), "retard", "cunt", "faggot", sexual or family insults, wishing harm or